In [2]:
from datasets import load_dataset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
import spacy
import re
from collections import Counter
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except:
    print("⚠️ SpaCy model not found. Run: python -m spacy download en_core_web_sm")
    nlp = None

# Set random seed
np.random.seed(42)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/abdul-razzaq-munshi/Desktop/mLlabproject/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/abdul-razzaq-munshi/Desktop/mLlabproject/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/abdul-razzaq-munshi/Desktop/mLlabproject/venv/lib/python3.12/site-packages/i

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
import datasets
print(datasets.__version__)
print(datasets.__file__)


ImportError: cannot import name 'BeamBasedBuilder' from 'datasets.builder' (c:\ML_proj\.venv\Lib\site-packages\datasets\builder.py)

In [ ]:
import sys
print(sys.executable)


c:\ML_proj\.venv\Scripts\python.exe


In [ ]:
class Config:
    # Data parameters
    SAMPLE_SIZE = 5000  # Match transformer approach
    TEST_SIZE = 0.2
    VALIDATION_SIZE = 0.1
    RANDOM_SEED = 42
    
    # Feature extraction
    USE_PERPLEXITY = True
    MAX_TEXT_LENGTH = 512  # For perplexity calculation
    
    # Model selection
    MODELS = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'SVM': SVC(kernel='rbf', probability=True, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }
    
    # Output
    RESULTS_FILE = "feature_ml_results.json"
    FEATURES_FILE = "extracted_features.csv"

config = Config()


In [ ]:
print("📦 Loading dataset...")
dataset = load_dataset("andythetechnerd03/AI-human-text")

# Use subset for faster processing
print(f"🎯 Using subset of {config.SAMPLE_SIZE} samples...")
dataset["train"] = dataset["train"].shuffle(seed=config.RANDOM_SEED).select(
    range(min(config.SAMPLE_SIZE, len(dataset["train"])))
)

# Check and rename label column
if 'label' not in dataset['train'].column_names:
    possible_label_cols = ['labels', 'target', 'class', 'category', 'generated']
    label_col = None
    for col in possible_label_cols:
        if col in dataset['train'].column_names:
            label_col = col
            break
    if label_col:
        dataset = dataset.rename_column(label_col, 'label')

print(f"Dataset size: {len(dataset['train'])}")
print(f"Sample: {dataset['train'][0]}\n")


In [ ]:
print("🔧 Setting up feature extractors...")

# Load GPT-2 for perplexity calculation
if config.USE_PERPLEXITY:
    print("Loading GPT-2 for perplexity calculation...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
    gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
    gpt2_model.eval()
    print(f"GPT-2 loaded on {device}")

def calculate_perplexity(text, max_length=512):
    """Calculate perplexity using GPT-2"""
    if not config.USE_PERPLEXITY:
        return 0.0
    
    try:
        encodings = gpt2_tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length)
        input_ids = encodings.input_ids.to(device)
        
        with torch.no_grad():
            outputs = gpt2_model(input_ids, labels=input_ids)
            loss = outputs.loss
            perplexity = torch.exp(loss).item()
        
        return perplexity
    except:
        return 0.0

def extract_text_features(text):
    """Extract comprehensive linguistic features from text"""
    features = {}
    
    # Basic text stats
    features['text_length'] = len(text)
    features['char_count'] = len(text)
    
    # Sentence-level features
    sentences = sent_tokenize(text)
    features['sentence_count'] = len(sentences)
    features['avg_sentence_length'] = np.mean([len(s.split()) for s in sentences]) if sentences else 0
    features['max_sentence_length'] = max([len(s.split()) for s in sentences]) if sentences else 0
    features['min_sentence_length'] = min([len(s.split()) for s in sentences]) if sentences else 0
    features['sentence_length_std'] = np.std([len(s.split()) for s in sentences]) if sentences else 0
    
    # Word-level features
    words = word_tokenize(text.lower())
    features['word_count'] = len(words)
    features['avg_word_length'] = np.mean([len(w) for w in words]) if words else 0
    features['max_word_length'] = max([len(w) for w in words]) if words else 0
    
    # Vocabulary richness
    unique_words = set(words)
    features['unique_word_count'] = len(unique_words)
    features['lexical_diversity'] = len(unique_words) / len(words) if words else 0
    
    # Character-level features
    features['uppercase_ratio'] = sum(1 for c in text if c.isupper()) / len(text) if text else 0
    features['digit_ratio'] = sum(1 for c in text if c.isdigit()) / len(text) if text else 0
    features['punctuation_ratio'] = sum(1 for c in text if c in '.,!?;:') / len(text) if text else 0
    features['space_ratio'] = sum(1 for c in text if c.isspace()) / len(text) if text else 0
    
    # POS tag distribution
    try:
        pos_tags = pos_tag(words)
        pos_counts = Counter([tag for _, tag in pos_tags])
        total_tags = len(pos_tags)
        
        # Most common POS tags
        features['noun_ratio'] = sum(pos_counts[tag] for tag in pos_counts if tag.startswith('NN')) / total_tags if total_tags else 0
        features['verb_ratio'] = sum(pos_counts[tag] for tag in pos_counts if tag.startswith('VB')) / total_tags if total_tags else 0
        features['adj_ratio'] = sum(pos_counts[tag] for tag in pos_counts if tag.startswith('JJ')) / total_tags if total_tags else 0
        features['adv_ratio'] = sum(pos_counts[tag] for tag in pos_counts if tag.startswith('RB')) / total_tags if total_tags else 0
        features['pronoun_ratio'] = sum(pos_counts[tag] for tag in pos_counts if tag.startswith('PR')) / total_tags if total_tags else 0
        features['determiner_ratio'] = pos_counts.get('DT', 0) / total_tags if total_tags else 0
        features['pos_diversity'] = len(pos_counts) / total_tags if total_tags else 0
    except:
        features.update({
            'noun_ratio': 0, 'verb_ratio': 0, 'adj_ratio': 0,
            'adv_ratio': 0, 'pronoun_ratio': 0, 'determiner_ratio': 0,
            'pos_diversity': 0
        })
    
    # SpaCy features (if available)
    if nlp:
        try:
            doc = nlp(text[:1000000])  # Limit for performance
            features['entity_count'] = len(doc.ents)
            features['entity_density'] = len(doc.ents) / len(doc) if len(doc) > 0 else 0
        except:
            features['entity_count'] = 0
            features['entity_density'] = 0
    else:
        features['entity_count'] = 0
        features['entity_density'] = 0
    
    # Punctuation patterns
    features['comma_count'] = text.count(',')
    features['period_count'] = text.count('.')
    features['exclamation_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['semicolon_count'] = text.count(';')
    features['colon_count'] = text.count(':')
    
    # Readability (simple heuristics)
    features['avg_words_per_sentence'] = features['word_count'] / features['sentence_count'] if features['sentence_count'] else 0
    features['avg_chars_per_word'] = features['char_count'] / features['word_count'] if features['word_count'] else 0
    
    # Perplexity (computationally expensive - last)
    if config.USE_PERPLEXITY:
        features['perplexity'] = calculate_perplexity(text, config.MAX_TEXT_LENGTH)
    
    return features


In [ ]:
print("\n🔍 Extracting features from all texts...")
print("This may take a few minutes...")

all_features = []
all_labels = []

for i, example in enumerate(dataset['train']):
    if (i + 1) % 500 == 0:
        print(f"  Processed {i + 1}/{len(dataset['train'])} samples...")
    
    features = extract_text_features(example['text'])
    all_features.append(features)
    all_labels.append(example['label'])

# Convert to DataFrame
print("\n📊 Creating feature matrix...")
X = pd.DataFrame(all_features)
y = np.array(all_labels)

print(f"Feature matrix shape: {X.shape}")
print(f"\nExtracted features ({len(X.columns)}):")
print(X.columns.tolist())

# Save features
X['label'] = y
X.to_csv(config.FEATURES_FILE, index=False)
print(f"\n💾 Features saved to '{config.FEATURES_FILE}'")
X = X.drop('label', axis=1)

# Handle any NaN or inf values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

print(f"\nFeature statistics:")
print(X.describe())


In [ ]:
print("\n🧹 Splitting data...")

# First split: train+val vs test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_SEED, stratify=y
)

# Second split: train vs val
val_size_adjusted = config.VALIDATION_SIZE / (1 - config.TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size_adjusted, random_state=config.RANDOM_SEED, stratify=y_temp
)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")


In [ ]:
print("\n⚙️ Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print("\n🚀 Training models...\n")

results = {}
trained_models = {}

for model_name, model in config.MODELS.items():
    print(f"{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict on validation set
    y_val_pred = model.predict(X_val_scaled)
    y_val_proba = model.predict_proba(X_val_scaled)[:, 1] if hasattr(model, 'predict_proba') else y_val_pred
    
    # Predict on test set
    y_test_pred = model.predict(X_test_scaled)
    y_test_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else y_test_pred
    
    # Calculate metrics
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1_weighted = f1_score(y_test, y_test_pred, average='weighted')
    test_f1_per_class = f1_score(y_test, y_test_pred, average=None)
    test_precision = precision_score(y_test, y_test_pred, average='weighted')
    test_recall = recall_score(y_test, y_test_pred, average='weighted')
    
    # AUC metrics
    try:
        fpr, tpr, _ = roc_curve(y_test, y_test_proba)
        test_auc_roc = auc(fpr, tpr)
        test_pr_auc = average_precision_score(y_test, y_test_proba)
    except:
        test_auc_roc = 0.0
        test_pr_auc = 0.0
    
    results[model_name] = {
        'model': model,
        'y_test_pred': y_test_pred,
        'y_test_proba': y_test_proba,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'f1_weighted': test_f1_weighted,
        'f1_human': test_f1_per_class[0],
        'f1_ai': test_f1_per_class[1],
        'precision': test_precision,
        'recall': test_recall,
        'auc_roc': test_auc_roc,
        'pr_auc': test_pr_auc
    }
    
    trained_models[model_name] = model
    
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"F1 Score: {test_f1_weighted:.4f}")
    print(f"AUC-ROC: {test_auc_roc:.4f}\n")

# Find best model
best_model_name = max(results.keys(), key=lambda k: results[k]['f1_weighted'])
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   F1 Score: {results[best_model_name]['f1_weighted']:.4f}")


In [ ]:
print("\n📊 Analyzing feature importance...")

fig_importance, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

importance_data = []

for idx, (model_name, result) in enumerate(results.items()):
    model = result['model']
    
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        importance_type = 'Feature Importance'
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])
        importance_type = 'Coefficient Magnitude'
    else:
        continue
    
    # Get top 15 features
    indices = np.argsort(importances)[-15:]
    top_features = X.columns[indices]
    top_importances = importances[indices]
    
    # Plot
    axes[idx].barh(range(len(top_features)), top_importances)
    axes[idx].set_yticks(range(len(top_features)))
    axes[idx].set_yticklabels(top_features)
    axes[idx].set_xlabel(importance_type)
    axes[idx].set_title(f'{model_name} - Top 15 Features', fontweight='bold')
    axes[idx].grid(axis='x', alpha=0.3)
    
    # Store for comparison
    for feat, imp in zip(top_features, top_importances):
        importance_data.append({
            'model': model_name,
            'feature': feat,
            'importance': imp
        })

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print(f"\n📈 Creating visualizations for best model: {best_model_name}")

best_result = results[best_model_name]
y_pred = best_result['y_test_pred']
y_proba = best_result['y_test_proba']

fig = plt.figure(figsize=(18, 12))

# 1. Confusion Matrix
ax1 = plt.subplot(2, 3, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
plt.title(f'Confusion Matrix\n{best_model_name}', fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# 2. Normalized Confusion Matrix
ax2 = plt.subplot(2, 3, 2)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
plt.title('Normalized Confusion Matrix', fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# 3. ROC Curve
ax3 = plt.subplot(2, 3, 3)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve', fontsize=12, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# 4. Precision-Recall Curve
ax4 = plt.subplot(2, 3, 4)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)
plt.plot(recall_vals, precision_vals, color='darkgreen', lw=2, 
         label=f'PR (AP = {avg_precision:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve', fontsize=12, fontweight='bold')
plt.legend(loc="lower left")
plt.grid(alpha=0.3)

# 5. Prediction Confidence Distribution
ax5 = plt.subplot(2, 3, 5)
human_probs = y_proba[y_test == 0]
ai_probs = y_proba[y_test == 1]
plt.hist(human_probs, bins=30, alpha=0.6, label='Human (True)', color='blue', edgecolor='black')
plt.hist(ai_probs, bins=30, alpha=0.6, label='AI (True)', color='red', edgecolor='black')
plt.xlabel('Predicted Probability (AI class)')
plt.ylabel('Frequency')
plt.title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

# 6. Model Comparison
ax6 = plt.subplot(2, 3, 6)
model_names = list(results.keys())
f1_scores = [results[name]['f1_weighted'] for name in model_names]
colors = ['#2ecc71' if name == best_model_name else '#3498db' for name in model_names]

bars = plt.bar(range(len(model_names)), f1_scores, color=colors)
plt.xticks(range(len(model_names)), model_names, rotation=45, ha='right')
plt.ylabel('F1 Score (Weighted)')
plt.title('Model Comparison', fontsize=12, fontweight='bold')
plt.ylim([0, 1.05])
plt.grid(axis='y', alpha=0.3)

for i, (bar, score) in enumerate(zip(bars, f1_scores)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('feature_ml_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
print(f"\n📋 Classification Report ({best_model_name}):")
print(classification_report(y_test, y_pred, target_names=['Human', 'AI'], digits=4))

In [ ]:
print("\n🔍 Error Analysis:")
errors = []
for i, (true, pred, prob) in enumerate(zip(y_test, y_pred, y_proba)):
    if true != pred:
        errors.append({
            'index': i,
            'true_label': 'AI' if true == 1 else 'Human',
            'pred_label': 'AI' if pred == 1 else 'Human',
            'confidence': prob if pred == 1 else 1 - prob
        })

print(f"Total errors: {len(errors)} out of {len(y_test)} ({len(errors)/len(y_test)*100:.2f}%)")


In [ ]:
print("\n📊 All Models - Results Summary:")

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['test_accuracy'] for m in results.keys()],
    'F1 (Weighted)': [results[m]['f1_weighted'] for m in results.keys()],
    'F1 (Human)': [results[m]['f1_human'] for m in results.keys()],
    'F1 (AI)': [results[m]['f1_ai'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'AUC-ROC': [results[m]['auc_roc'] for m in results.keys()],
    'PR-AUC': [results[m]['pr_auc'] for m in results.keys()]
})

print(comparison_df.to_string(index=False))

# Save to CSV
comparison_df.to_csv('feature_ml_comparison.csv', index=False)
print("\n💾 Results saved to 'feature_ml_comparison.csv'")


In [ ]:
detailed_results = {
    'approach': 'Feature-Based ML',
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'config': {
        'sample_size': config.SAMPLE_SIZE,
        'test_size': config.TEST_SIZE,
        'num_features': X.shape[1],
        'use_perplexity': config.USE_PERPLEXITY
    },
    'dataset_info': {
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test)
    },
    'best_model': {
        'name': best_model_name,
        'metrics': {
            'accuracy': float(results[best_model_name]['test_accuracy']),
            'f1_weighted': float(results[best_model_name]['f1_weighted']),
            'f1_human': float(results[best_model_name]['f1_human']),
            'f1_ai': float(results[best_model_name]['f1_ai']),
            'precision': float(results[best_model_name]['precision']),
            'recall': float(results[best_model_name]['recall']),
            'auc_roc': float(results[best_model_name]['auc_roc']),
            'pr_auc': float(results[best_model_name]['pr_auc'])
        }
    },
    'all_models': {
        name: {
            'accuracy': float(res['test_accuracy']),
            'f1_weighted': float(res['f1_weighted']),
            'auc_roc': float(res['auc_roc'])
        }
        for name, res in results.items()
    },
    'confusion_matrix': cm.tolist(),
    'error_count': len(errors),
    'feature_names': X.columns.tolist()
}

with open(config.RESULTS_FILE, 'w') as f:
    json.dump(detailed_results, f, indent=2)

print(f"💾 Detailed results saved to '{config.RESULTS_FILE}'")

In [ ]:
def predict_new_text(text, model_name=best_model_name, return_confidence=False):
    """
    Predict if text is AI-generated or Human-written using feature-based ML
    
    Args:
        text: Input text string
        model_name: Which trained model to use
        return_confidence: If True, returns (label, confidence) tuple
    
    Returns:
        Label string or (label, confidence) tuple
    """
    # Extract features
    features = extract_text_features(text)
    feature_vector = pd.DataFrame([features])[X.columns]
    feature_vector = feature_vector.fillna(0)
    
    # Scale features
    feature_scaled = scaler.transform(feature_vector)
    
    # Predict
    model = trained_models[model_name]
    pred = model.predict(feature_scaled)[0]
    
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(feature_scaled)[0]
        confidence = proba[pred]
    else:
        confidence = 1.0
    
    label = "AI-generated" if pred == 1 else "Human-written"
    
    if return_confidence:
        return label, confidence
    return label


In [ ]:
print(f"\n💬 Testing inference with {best_model_name}:")
test_examples = [
    "Artificial intelligence has revolutionized modern computing with advanced algorithms.",
    "I spent my weekend hiking in the mountains with friends and it was amazing!",
    "The implementation of neural networks requires careful consideration of hyperparameters.",
    "Yesterday I made the best pasta ever. The secret is lots of garlic and fresh basil."
]

for text in test_examples:
    label, conf = predict_new_text(text, return_confidence=True)
    print(f"\nText: '{text[:80]}...'")
    print(f"Prediction: {label} (confidence: {conf:.3f})")

print("\n" + "="*80)
print("🎉 Feature-Based ML analysis complete!")
print("="*80)
print(f"\nBest performing model: {best_model_name}")
print(f"Test F1 Score: {results[best_model_name]['f1_weighted']:.4f}")
print(f"Test Accuracy: {results[best_model_name]['test_accuracy']:.4f}")
print(f"AUC-ROC: {results[best_model_name]['auc_roc']:.4f}")